# A/A Test: Customer Lookalike Methodology Validation

## Purpose

This notebook validates the **Customer Lookalike synthetic control methodology** by running it against historical periods where **no campaign was active**. The expected result is **zero uplift** and **zero incremental orders**. Any non-zero result quantifies the methodology's inherent measurement bias.

## Methodology

1. Select customers from a market (e.g., DE) during a confirmed non-campaign period
2. Randomly split them into treatment (1) and potential lookalike (0) — **no actual treatment is applied**
3. Run the exact same matching logic as the production scripts (exact match + KNN fallback)
4. Calculate uplift metrics — all should be ~0%
5. Repeat across multiple time windows and random seeds to assess stability

## Script Variants Tested

- **V3** (`customer_look_alike_evaluation_v3_with_promo_orders_bilal.py`): Primary test. Includes promo order correction factor.
- **V2** (`customer_look_alike_evaluation_v2_with_city_type.py`): Optional. Adds `city_type` matching dimension.

## Pass/Fail Criteria

| Verdict | Condition |
|---------|----------|
| **PASS** | \|mean uplift\| ≤ 2% AND 95% CI contains zero |
| **WARNING** | \|mean uplift\| > 2% OR 95% CI excludes zero |
| **HARD FAIL** | \|mean uplift\| > 5% in any period |

Results are reported at **total** level and **per customer base segment** (New, Reactivated, Early, Engaged, Lapsing, Dormant, Prospect, Churned, unknown).

---
## 1. Imports & Authentication

In [ ]:
import pandas as pd
import numpy as np
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import MinMaxScaler
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns
import time
import warnings

from google.colab import auth
from google.cloud import bigquery

warnings.filterwarnings('ignore', category=FutureWarning)
sns.set_style('whitegrid')

auth.authenticate_user()
print('Authenticated')

PROJECT_ID = 'just-data-gci-dev'
client = bigquery.Client(project=PROJECT_ID)
print(f'Connected to project: {PROJECT_ID}')

---
## 2. Configuration

In [ ]:
# ============================================================
# A/A TEST CONFIGURATION
# ============================================================

# Markets to test
COUNTRIES = ['DE']

# Number of random treatment/control splits per time window
N_SEEDS = 10

# Proportion assigned to treatment group
TREATMENT_SHARE = 0.5

# Bias thresholds for pass/fail
BIAS_THRESHOLD_HARD = 0.05  # |uplift| > 5% = HARD FAIL
BIAS_THRESHOLD_WARN = 0.02  # |uplift| > 2% = WARNING

# Campaign name prefix for A/A test runs
AA_CAMPAIGN_PREFIX = 'AA_CUSTOMER_TEST'

# Customer base segments — results are broken down per segment
BASE_SEGMENTS = [
    'New',
    'Reactivated',
    'Early',
    'Engaged',
    'Lapsing',
    'Dormant',
    'Prospect',
    'Churned',
    'unknown',
]

# Set to a list to test specific segments only, e.g. ['Engaged', 'Dormant']
# Set to None to include all segments
SEGMENTS_TO_TEST = None

# Non-overlapping 30-day dummy windows where NO real campaign was active
# IMPORTANT: validate these against the campaign calendar before running
TIME_WINDOWS = [
    {'start': '2024-06-01', 'end': '2024-06-30',
     'post_start': '2024-07-01', 'post_end': '2024-07-31'},
    {'start': '2024-08-01', 'end': '2024-08-31',
     'post_start': '2024-09-01', 'post_end': '2024-09-30'},
    {'start': '2024-10-01', 'end': '2024-10-31',
     'post_start': '2024-11-01', 'post_end': '2024-11-30'},
    {'start': '2025-02-01', 'end': '2025-02-28',
     'post_start': '2025-03-01', 'post_end': '2025-03-31'},
]

# V3 matching conditions (from production script)
LOOK_ALIKE_CONDITIONS_V3 = [
    'country', 'optin', 'campaign_start_date',
    'L14D_orders', 'L30D_orders', 'L90D_orders', 'L365D_orders'
]

# V2 matching conditions (from production script)
LOOK_ALIKE_CONDITIONS_V2 = [
    'country', 'city_type', 'optin', 'campaign_start_date',
    'L14D_orders', 'L30D_orders', 'L90D_orders', 'L365D_orders', 'L365D_AOV_cat'
]

# Dedicated A/A tables — avoids touching production tables
AA_AUDIENCE_TABLE = 'just-data-warehouse.customer_intelligence.customer_lookalike_aa_audience'
AA_RESULTS_TABLE = 'just-data-warehouse.customer_intelligence.customer_lookalike_aa_results'

print('Configuration loaded.')
print(f'Markets: {COUNTRIES}')
print(f'Time windows: {len(TIME_WINDOWS)}')
print(f'Seeds per window: {N_SEEDS}')
print(f'Total runs: {len(COUNTRIES) * len(TIME_WINDOWS) * N_SEEDS}')
print(f'Segments: {SEGMENTS_TO_TEST or "all"}')

---
## 3. Schema Discovery & Campaign Overlap Validation

### 3.1 Discover `base_segment` column in `dim_customer`

The base segment column name may vary (`base_segment`, `customer_segment`, `lifecycle_stage`, etc.). Run this cell to inspect the schema.

In [ ]:
# Inspect dim_customer schema to find the segment column
schema_query = """
SELECT column_name, data_type
FROM `just-data-warehouse.dwh.INFORMATION_SCHEMA.COLUMNS`
WHERE table_name = 'dim_customer'
  AND LOWER(column_name) LIKE '%segment%'
   OR LOWER(column_name) LIKE '%lifecycle%'
   OR LOWER(column_name) LIKE '%base%'
ORDER BY column_name
"""
df_schema = client.query(schema_query).to_dataframe()
print('Candidate segment columns in dim_customer:')
df_schema

In [ ]:
# ============================================================
# SET THIS AFTER INSPECTING THE SCHEMA ABOVE
# ============================================================
SEGMENT_COLUMN = 'base_segment'  # <-- Update if the column name differs

# Verify: show distinct segment values
verify_query = f"""
SELECT DISTINCT {SEGMENT_COLUMN} AS segment, COUNT(*) AS cnt
FROM `just-data-warehouse.dwh.dim_customer`
WHERE {SEGMENT_COLUMN} IS NOT NULL
GROUP BY 1
ORDER BY 2 DESC
LIMIT 20
"""
df_segments = client.query(verify_query).to_dataframe()
print(f'Distinct values in {SEGMENT_COLUMN}:')
df_segments

### 3.2 Validate no campaign overlap with dummy windows

In [ ]:
def validate_no_campaign_overlap(client, countries, time_windows):
    """Check that no real campaigns overlap with the chosen A/A test windows."""
    checks = []
    for country in countries:
        for w in time_windows:
            # Check production V3 audience table
            for table in [
                'just-data-warehouse.customer_intelligence.customer_lookalike_evaluation_audience',
                'just-data-warehouse.customer_intelligence.customer_lookalike_evaluation_audience_citytype',
            ]:
                query = f"""
                SELECT COUNT(*) as cnt
                FROM `{table}`
                WHERE country = '{country}'
                  AND (
                    (campaign_start_date BETWEEN '{w['start']}' AND '{w['post_end']}')
                    OR (campaign_end_date BETWEEN '{w['start']}' AND '{w['post_end']}')
                  )
                """
                try:
                    result = client.query(query).to_dataframe()
                    cnt = result['cnt'].iloc[0]
                except Exception as e:
                    cnt = -1  # table may not exist
                checks.append({
                    'country': country,
                    'window': f"{w['start']} to {w['post_end']}",
                    'table': table.split('.')[-1],
                    'overlapping_rows': cnt,
                })

    df_checks = pd.DataFrame(checks)
    overlaps = df_checks[df_checks['overlapping_rows'] > 0]
    if len(overlaps) > 0:
        print('WARNING: Campaign overlap detected! Review these windows before proceeding:')
        display(overlaps)
    else:
        print('PASS: No campaign overlap detected for any window.')
    return df_checks

df_overlap = validate_no_campaign_overlap(client, COUNTRIES, TIME_WINDOWS)
df_overlap

---
## 4. A/A Audience Creation

In [ ]:
def create_aa_audience(client, country, window, seed,
                       treatment_share=0.5, segment_col='base_segment',
                       segments_filter=None):
    """
    Select all eligible customers for a country/window, then randomly split
    into treatment (1) and potential lookalike (0).

    Uses the same join pattern as the production scripts:
    fact_order + dim_customer (optin) + dim_country (countryid -> country).

    Adds base_segment for per-segment analysis.
    """
    start = window['start']

    segment_filter_sql = ''
    if segments_filter:
        segment_list = ', '.join([f"'{s}'" for s in segments_filter])
        segment_filter_sql = f"AND COALESCE(dcu.{segment_col}, 'unknown') IN ({segment_list})"

    query = f"""
    SELECT
        fo.customerid,
        dc.country,
        CASE WHEN dcu.optin = true THEN 1 ELSE 0 END AS optin,
        COALESCE(dcu.{segment_col}, 'unknown') AS base_segment
    FROM `just-data-warehouse.dwh.fact_order` AS fo
    INNER JOIN `just-data-warehouse.dwh.dim_customer` AS dcu
        ON fo.customerid = dcu.customerid
    INNER JOIN `just-data-warehouse.dwh.dim_country` AS dc
        ON fo.countryid = dc.countryid
    WHERE dc.country = '{country}'
      AND DATE(orderdatetime) >= DATE_SUB(DATE('{start}'), INTERVAL 365 DAY)
      AND DATE(orderdatetime) < DATE('{start}')
      AND DATE(orderdatetime) >= DATE('2021-01-01')
      {segment_filter_sql}
    GROUP BY 1, 2, 3, 4
    HAVING SUM(nroforders) > 0
    """

    df = client.query(query).to_dataframe()

    # Stratified random split by segment to ensure balanced groups
    rng = np.random.RandomState(seed)
    df['treatment'] = 0
    for segment in df['base_segment'].unique():
        mask = df['base_segment'] == segment
        n_segment = mask.sum()
        df.loc[mask, 'treatment'] = rng.binomial(1, treatment_share, size=n_segment)

    # Add campaign window metadata (common date for all customers)
    campaign_name = f"{AA_CAMPAIGN_PREFIX}_{country}_{start}_seed{seed}"
    df['campaign_name'] = campaign_name
    df['campaign_start_date'] = pd.to_datetime(window['start']).date()
    df['campaign_end_date'] = pd.to_datetime(window['end']).date()
    df['post_period_start_date'] = pd.to_datetime(window['post_start']).date()
    df['post_period_end_date'] = pd.to_datetime(window['post_end']).date()

    n_treat = (df['treatment'] == 1).sum()
    n_ctrl = (df['treatment'] == 0).sum()
    print(f'  {country} | {start} | seed {seed}: '
          f'{len(df)} customers ({n_treat} treatment, {n_ctrl} lookalike), '
          f'{df["base_segment"].nunique()} segments')

    return df, campaign_name


def upload_aa_audience(client, df, table_id):
    """Write A/A audience to a dedicated BigQuery table."""
    # Select only the columns matching the audience table schema
    upload_cols = [
        'customerid', 'country', 'campaign_name', 'treatment',
        'campaign_start_date', 'campaign_end_date',
        'post_period_start_date', 'post_period_end_date',
        'optin', 'base_segment',
    ]
    df_upload = df[upload_cols].copy()

    schema = [
        bigquery.SchemaField('customerid', 'STRING'),
        bigquery.SchemaField('country', 'STRING'),
        bigquery.SchemaField('campaign_name', 'STRING'),
        bigquery.SchemaField('treatment', 'INTEGER'),
        bigquery.SchemaField('campaign_start_date', 'DATE'),
        bigquery.SchemaField('campaign_end_date', 'DATE'),
        bigquery.SchemaField('post_period_start_date', 'DATE'),
        bigquery.SchemaField('post_period_end_date', 'DATE'),
        bigquery.SchemaField('optin', 'INTEGER'),
        bigquery.SchemaField('base_segment', 'STRING'),
    ]
    job_config = bigquery.LoadJobConfig(
        schema=schema,
        write_disposition='WRITE_APPEND',
    )
    job = client.load_table_from_dataframe(df_upload, table_id, job_config=job_config)
    job.result()
    print(f'  Uploaded {len(df_upload)} rows to {table_id}')


def cleanup_aa_audience(client, table_id, campaign_name):
    """Remove A/A audience rows for a specific run to keep the table clean."""
    query = f"""
    DELETE FROM `{table_id}`
    WHERE campaign_name = '{campaign_name}'
    """
    client.query(query).result()


# Safety: clean up any leftover rows from previous A/A runs
print('Cleaning up any leftover A/A audience rows...')
try:
    cleanup_query = f"""
    DELETE FROM `{AA_AUDIENCE_TABLE}`
    WHERE campaign_name LIKE '{AA_CAMPAIGN_PREFIX}%'
    """
    client.query(cleanup_query).result()
    print('Cleanup complete.')
except Exception as e:
    print(f'Table may not exist yet (will be created on first upload): {e}')

---
## 5. Matching Logic — V3 (Adapted from Production)

This is a direct adaptation of the matching logic from `customer_look_alike_evaluation_v3_with_promo_orders_bilal.py`.

**Only two changes from production:**
1. Table reference → `AA_AUDIENCE_TABLE`
2. Campaign name passed as parameter

All matching logic, outlier exclusion, NTILE bucketing, exact match, and KNN fallback are **identical** to production.

In [ ]:
def run_matching_v3(client, country, date, campaign_name, audience_table,
                    look_alike_conditions):
    """
    Run V3 Customer Lookalike matching for a single country/date cohort.

    Adapted from customer_look_alike_evaluation_v3_with_promo_orders_bilal.py
    lines 122-358. Only table references and campaign_name sourcing changed.
    """
    date_str = date if isinstance(date, str) else date.strftime('%Y-%m-%d')

    # Feature query — identical to production lines 122-157
    query = f"""
    SELECT
    *,
    NTILE(3) OVER (PARTITION BY country ORDER BY L365D_AOV) AS L365D_AOV_cat
    FROM (
        SELECT  fo.customerid,
                lc.country,
                CASE WHEN dcu.optin = true THEN 1 ELSE 0 END AS optin,
                lc.campaign_start_date,
                lc.campaign_end_date,
                '{campaign_name}' as campaign_name,
                lc.treatment,
                lc.base_segment,
                SUM(CASE WHEN DATE(orderdatetime) >= DATE_ADD(DATE(campaign_start_date), INTERVAL -7 DAY)  AND DATE(orderdatetime) < DATE(campaign_start_date) THEN nroforders ELSE 0 END) as L7D_orders,
                SUM(CASE WHEN DATE(orderdatetime) >= DATE_ADD(DATE(campaign_start_date), INTERVAL -14 DAY)  AND DATE(orderdatetime) < DATE(campaign_start_date) THEN nroforders ELSE 0 END) as L14D_orders,
                SUM(CASE WHEN DATE(orderdatetime) >= DATE_ADD(DATE(campaign_start_date), INTERVAL -30 DAY)  AND DATE(orderdatetime) < DATE(campaign_start_date) THEN nroforders ELSE 0 END) as L30D_orders,
                SUM(CASE WHEN DATE(orderdatetime) >= DATE_ADD(DATE(campaign_start_date), INTERVAL -90 DAY)  AND DATE(orderdatetime) < DATE(campaign_start_date) THEN nroforders ELSE 0 END) as L90D_orders,
                SUM(CASE WHEN DATE(orderdatetime) >= DATE_ADD(DATE(campaign_start_date), INTERVAL -30 DAY)  AND DATE(orderdatetime) < DATE(campaign_start_date) AND restaurant_discount > 0 THEN nroforders ELSE 0 END) as L30D_promo_orders,
                SUM(CASE WHEN DATE(orderdatetime) >= DATE_ADD(DATE(campaign_start_date), INTERVAL -180 DAY) AND DATE(orderdatetime) < DATE(campaign_start_date) THEN nroforders ELSE 0 END) as L180D_orders,
                SUM(CASE WHEN DATE(orderdatetime) >= DATE_ADD(DATE(campaign_start_date), INTERVAL -365 DAY) AND DATE(orderdatetime) < DATE(campaign_start_date) THEN nroforders ELSE 0 END) as L365D_orders,
                SUM(CASE WHEN DATE(orderdatetime) >= DATE_ADD(DATE(campaign_start_date), INTERVAL -730 DAY) AND DATE(orderdatetime) < DATE(campaign_start_date) THEN nroforders ELSE 0 END) as L730D_orders,
                SAFE_DIVIDE(SUM(CASE WHEN DATE(orderdatetime) >= DATE_ADD(DATE(campaign_start_date), INTERVAL -365 DAY) AND DATE(orderdatetime) < DATE(campaign_start_date) THEN food_total ELSE 0 END),
                SUM(CASE WHEN DATE(orderdatetime) >= DATE_ADD(DATE(campaign_start_date), INTERVAL -365 DAY) AND DATE(orderdatetime) < DATE(campaign_start_date) THEN nroforders ELSE 0 END)) as L365D_AOV,
                SUM(CASE WHEN DATE(orderdatetime) > DATE(campaign_start_date) AND DATE(orderdatetime) <= DATE(campaign_end_date) THEN nroforders ELSE 0 END) as campaign_period_orders,
                SUM(CASE WHEN DATE(orderdatetime) > DATE(campaign_start_date) AND DATE(orderdatetime) <= DATE(campaign_end_date) AND restaurant_discount > 0 THEN nroforders ELSE 0 END) as campaign_period_promo_orders,
                SUM(CASE WHEN DATE(orderdatetime) > DATE(post_period_start_date) AND DATE(orderdatetime) <= DATE(post_period_end_date) THEN nroforders ELSE 0 END) as post_period_orders
        FROM `just-data-warehouse.dwh.fact_order` as fo
                  INNER JOIN `just-data-warehouse.dwh.dim_customer` as dcu on fo.customerid = dcu.customerid
                  INNER JOIN `just-data-warehouse.dwh.dim_country` as dc on fo.countryid = dc.countryid
                  INNER JOIN `{audience_table}` as lc on fo.customerid = lc.customerid
        WHERE DATE(orderdatetime) >= DATE_ADD(DATE(campaign_start_date), INTERVAL -730 DAY)
          AND DATE(orderdatetime) >= DATE('2020-01-01')
          AND campaign_name = '{campaign_name}'
          AND lc.country = '{country}'
          AND DATE(campaign_start_date) = '{date_str}'
        GROUP BY 1, 2, 3, 4, 5, 6, 7, 8
        )
    """

    df = client.query(query).to_dataframe()
    if len(df) == 0:
        print(f'  WARNING: No data returned for {country} / {date_str}')
        return None
    print(f'  {len(df)} customers in {country} / {date_str} cohort')

    # Outlier exclusion — identical to production lines 165-187
    treatment = df[
        (df['L7D_orders'] <= 7) &
        (df['L14D_orders'] <= 14) &
        (df['L30D_orders'] <= 30) &
        (df['L90D_orders'] <= 90) &
        (df['L180D_orders'] <= 180) &
        (df['L365D_orders'] <= 365) &
        (df['L730D_orders'] <= 730) &
        (df['L30D_promo_orders'] == 0) &
        (df['treatment'] == 1)
    ].copy()

    lookalike = df[
        (df['L7D_orders'] <= 7) &
        (df['L14D_orders'] <= 14) &
        (df['L30D_orders'] <= 30) &
        (df['L90D_orders'] <= 90) &
        (df['L180D_orders'] <= 180) &
        (df['L365D_orders'] <= 365) &
        (df['L730D_orders'] <= 730) &
        (df['L30D_promo_orders'] == 0) &
        (df['treatment'] == 0)
    ].copy()

    n_excluded = len(df) - len(treatment) - len(lookalike)
    print(f'  Excluded {n_excluded} outliers. '
          f'Treatment: {len(treatment)}, Lookalike: {len(lookalike)}')

    if len(treatment) == 0 or len(lookalike) == 0:
        print(f'  WARNING: Empty treatment or lookalike group, skipping.')
        return None

    # --- Exact match --- (production lines 196-207)
    merged_df = pd.merge(
        treatment, lookalike,
        on=look_alike_conditions,
        suffixes=('_treatment', '_lookalike')
    )

    merged_df = pd.merge(
        merged_df,
        lookalike[['customerid', 'L30D_orders']],
        left_on='customerid_lookalike',
        right_on='customerid',
        how='left',
        suffixes=('_treatment', '_lookalike')
    )

    grouped_df = merged_df.groupby([
        'customerid_treatment', 'country', 'optin', 'campaign_start_date',
        'L30D_orders_treatment', 'campaign_period_orders_treatment',
        'campaign_period_promo_orders_treatment', 'post_period_orders_treatment',
        'base_segment_treatment',
    ])[[
        'L30D_orders_lookalike', 'campaign_period_orders_lookalike',
        'campaign_period_promo_orders_lookalike', 'post_period_orders_lookalike',
    ]].mean().reset_index()

    grouped_df = grouped_df.rename(columns={'base_segment_treatment': 'base_segment'})

    print(f'  Exact match: {len(grouped_df)} of {len(treatment)} customers')

    # --- KNN for unmatched --- (production lines 213-269)
    missing_ids = set(treatment['customerid']) - set(grouped_df['customerid_treatment'])
    unmatched_df = treatment[treatment['customerid'].isin(missing_ids)].copy()

    columns_to_scale = [
        'optin', 'L14D_orders', 'L30D_orders', 'L90D_orders',
        'L180D_orders', 'L365D_orders', 'L730D_orders'
    ]

    if not unmatched_df.empty and not lookalike.empty:
        scaler = MinMaxScaler()
        lookalike_scaled = lookalike.copy()
        unmatched_scaled = unmatched_df.copy()

        lookalike_scaled[columns_to_scale] = scaler.fit_transform(
            lookalike_scaled[columns_to_scale]
        )
        unmatched_scaled[columns_to_scale] = scaler.transform(
            unmatched_scaled[columns_to_scale]
        )

        # Filter for current country/date
        lookalike_subset = lookalike_scaled[
            (lookalike_scaled['country'] == country) &
            (lookalike_scaled['campaign_start_date'] == pd.to_datetime(date_str).date())
        ]
        unmatched_subset = unmatched_scaled[
            (unmatched_scaled['country'] == country) &
            (unmatched_scaled['campaign_start_date'] == pd.to_datetime(date_str).date())
        ]

        if not unmatched_subset.empty and not lookalike_subset.empty:
            knn = NearestNeighbors(n_neighbors=1)
            knn.fit(lookalike_subset[columns_to_scale])
            distances, indices = knn.kneighbors(unmatched_subset[columns_to_scale])

            nearest_neighbors_df = pd.DataFrame({
                'customerid_treatment': unmatched_subset['customerid'].values,
                'country': country,
                'optin': unmatched_subset['optin'].values,
                'campaign_start_date': pd.to_datetime(date_str).date(),
                'L30D_orders_treatment': unmatched_df.loc[unmatched_subset.index, 'L30D_orders'].values,
                'campaign_period_orders_treatment': unmatched_subset['campaign_period_orders'].values,
                'campaign_period_promo_orders_treatment': unmatched_subset['campaign_period_promo_orders'].values,
                'post_period_orders_treatment': unmatched_subset['post_period_orders'].values,
                'base_segment': unmatched_subset['base_segment'].values,
                'L30D_orders_lookalike': lookalike.loc[lookalike_subset.iloc[indices.flatten()].index, 'L30D_orders'].values,
                'campaign_period_orders_lookalike': lookalike_subset['campaign_period_orders'].iloc[indices.flatten()].values,
                'campaign_period_promo_orders_lookalike': lookalike_subset['campaign_period_promo_orders'].iloc[indices.flatten()].values,
                'post_period_orders_lookalike': lookalike_subset['post_period_orders'].iloc[indices.flatten()].values,
            })

            print(f'  KNN matched: {len(nearest_neighbors_df)} unmatched customers')
        else:
            nearest_neighbors_df = pd.DataFrame()
    else:
        nearest_neighbors_df = pd.DataFrame()

    # --- Combine exact match + KNN ---
    grouped_df['source'] = 'exact_match'

    if not nearest_neighbors_df.empty:
        nearest_neighbors_df['source'] = 'nearest_neighbor'
        final_df = pd.concat(
            [grouped_df.reset_index(drop=True),
             nearest_neighbors_df.reset_index(drop=True)],
            ignore_index=True
        )
    else:
        final_df = grouped_df.reset_index(drop=True)

    final_df['campaign_name'] = campaign_name
    final_df['run_timestamp'] = pd.Timestamp.now(tz='Europe/Amsterdam')

    print(f'  Total matched: {len(final_df)} of {len(treatment)} treatment customers')
    return final_df

---
## 6. Matching Logic — V2 (Optional, with `city_type`)

**Note:** V2 requires a `city_type` column in the audience table. If `city_type` is not available in `dim_customer`, this section should be skipped. Set `RUN_V2 = False` to skip.

In [ ]:
# Set to True to also run V2 matching (requires city_type in audience)
RUN_V2 = False

# TODO: If enabling V2, you need to:
# 1. Identify the source of city_type (check dim_customer or a separate table)
# 2. Add city_type to the audience creation query in Section 4
# 3. Add city_type to the AA_AUDIENCE_TABLE schema
# 4. Uncomment and adapt the V2 matching function below

if RUN_V2:
    print('V2 matching enabled — ensure city_type is populated in audience table.')
else:
    print('V2 matching skipped. Set RUN_V2 = True to enable.')

In [ ]:
def run_matching_v2(client, country, date, city_type, campaign_name,
                    audience_table, look_alike_conditions):
    """
    Run V2 Customer Lookalike matching for a single country/date/city_type cohort.

    Adapted from customer_look_alike_evaluation_v2_with_city_type.py lines 49-311.
    Only table references and campaign_name sourcing changed.

    NOTE: This function requires city_type in the audience table.
    """
    date_str = date if isinstance(date, str) else date.strftime('%Y-%m-%d')

    query = f"""
    SELECT
    *,
    NTILE(3) OVER (PARTITION BY country ORDER BY L365D_AOV) AS L365D_AOV_cat
    FROM (
        SELECT  fo.customerid,
                lc.country,
                lc.city_type,
                CASE WHEN dcu.optin = true THEN 1 ELSE 0 END AS optin,
                lc.campaign_start_date,
                lc.campaign_end_date,
                '{campaign_name}' as campaign_name,
                lc.treatment,
                lc.base_segment,
                SUM(CASE WHEN DATE(orderdatetime) >= DATE_ADD(DATE(campaign_start_date), INTERVAL -7 DAY)  AND DATE(orderdatetime) < DATE(campaign_start_date) THEN nroforders ELSE 0 END) as L7D_orders,
                SUM(CASE WHEN DATE(orderdatetime) >= DATE_ADD(DATE(campaign_start_date), INTERVAL -14 DAY)  AND DATE(orderdatetime) < DATE(campaign_start_date) THEN nroforders ELSE 0 END) as L14D_orders,
                SUM(CASE WHEN DATE(orderdatetime) >= DATE_ADD(DATE(campaign_start_date), INTERVAL -30 DAY)  AND DATE(orderdatetime) < DATE(campaign_start_date) THEN nroforders ELSE 0 END) as L30D_orders,
                SUM(CASE WHEN DATE(orderdatetime) >= DATE_ADD(DATE(campaign_start_date), INTERVAL -30 DAY)  AND DATE(orderdatetime) < DATE(campaign_start_date) THEN CAST(ROUND(food_total) AS INT64) ELSE 0 END) as L30D_GMV,
                SUM(CASE WHEN DATE(orderdatetime) >= DATE_ADD(DATE(campaign_start_date), INTERVAL -90 DAY)  AND DATE(orderdatetime) < DATE(campaign_start_date) THEN nroforders ELSE 0 END) as L90D_orders,
                SUM(CASE WHEN DATE(orderdatetime) >= DATE_ADD(DATE(campaign_start_date), INTERVAL -180 DAY) AND DATE(orderdatetime) < DATE(campaign_start_date) THEN nroforders ELSE 0 END) as L180D_orders,
                SUM(CASE WHEN DATE(orderdatetime) >= DATE_ADD(DATE(campaign_start_date), INTERVAL -365 DAY) AND DATE(orderdatetime) < DATE(campaign_start_date) THEN nroforders ELSE 0 END) as L365D_orders,
                SAFE_DIVIDE(SUM(CASE WHEN DATE(orderdatetime) >= DATE_ADD(DATE(campaign_start_date), INTERVAL -365 DAY) AND DATE(orderdatetime) < DATE(campaign_start_date) THEN food_total ELSE 0 END),
                SUM(CASE WHEN DATE(orderdatetime) >= DATE_ADD(DATE(campaign_start_date), INTERVAL -365 DAY) AND DATE(orderdatetime) < DATE(campaign_start_date) THEN nroforders ELSE 0 END)) as L365D_AOV,
                SUM(CASE WHEN DATE(orderdatetime) > DATE(campaign_start_date) AND DATE(orderdatetime) <= DATE(campaign_end_date) THEN nroforders ELSE 0 END) as campaign_period_orders,
                SUM(CASE WHEN DATE(orderdatetime) > DATE(campaign_start_date) AND DATE(orderdatetime) <= DATE(campaign_end_date) THEN CAST(ROUND(restaurant_discount) AS INT64) ELSE 0 END) as campaign_period_discount,
                SUM(CASE WHEN DATE(orderdatetime) > DATE(campaign_start_date) AND DATE(orderdatetime) <= DATE(campaign_end_date) THEN CAST(ROUND(food_total) AS INT64) ELSE 0 END) as campaign_period_food_total,
                SUM(CASE WHEN DATE(orderdatetime) > DATE(post_period_start_date) AND DATE(orderdatetime) <= DATE(post_period_end_date) THEN nroforders ELSE 0 END) as post_period_orders
        FROM `just-data-warehouse.dwh.fact_order` as fo
                  INNER JOIN `just-data-warehouse.dwh.dim_customer` as dcu on fo.customerid = dcu.customerid
                  INNER JOIN `just-data-warehouse.dwh.dim_country` as dc on fo.countryid = dc.countryid
                  INNER JOIN `{audience_table}` as lc on fo.customerid = lc.customerid
        WHERE DATE(orderdatetime) >= DATE_ADD(DATE(campaign_start_date), INTERVAL -365 DAY)
          AND DATE(orderdatetime) >= DATE('2023-01-01')
          AND campaign_name = '{campaign_name}'
          AND lc.country = '{country}'
          AND lc.city_type = '{city_type}'
          AND DATE(campaign_start_date) = '{date_str}'
        GROUP BY 1, 2, 3, 4, 5, 6, 7, 8, 9
        )
    """

    df = client.query(query).to_dataframe()
    if len(df) == 0:
        return None

    # Outlier exclusion — V2 uses same hard limits, no promo filter
    treatment = df[
        (df['L7D_orders'] <= 7) & (df['L14D_orders'] <= 14) &
        (df['L30D_orders'] <= 30) & (df['L90D_orders'] <= 90) &
        (df['L180D_orders'] <= 180) & (df['L365D_orders'] <= 365) &
        (df['L365D_orders'] > 0) & (df['treatment'] == 1)
    ]
    lookalike = df[
        (df['L7D_orders'] <= 7) & (df['L14D_orders'] <= 14) &
        (df['L30D_orders'] <= 30) & (df['L90D_orders'] <= 90) &
        (df['L180D_orders'] <= 180) & (df['L365D_orders'] <= 365) &
        (df['L365D_orders'] > 0) & (df['treatment'] == 0)
    ]

    if len(treatment) == 0 or len(lookalike) == 0:
        return None

    # Exact match
    merged_df = pd.merge(treatment, lookalike, on=look_alike_conditions,
                         suffixes=('_treatment', '_lookalike'))

    merged_df = pd.merge(merged_df, lookalike[['customerid', 'L30D_orders', 'L30D_GMV']],
                         left_on='customerid_lookalike', right_on='customerid',
                         how='left', suffixes=('_treatment', '_lookalike'))

    grouped_df = merged_df.groupby([
        'customerid_treatment', 'country', 'city_type', 'optin',
        'campaign_start_date', 'L30D_orders_treatment', 'L30D_GMV_treatment',
        'campaign_period_orders_treatment', 'campaign_period_discount_treatment',
        'campaign_period_food_total_treatment', 'post_period_orders_treatment',
        'base_segment_treatment',
    ])[[
        'L30D_orders_lookalike', 'L30D_GMV_lookalike',
        'campaign_period_orders_lookalike', 'campaign_period_discount_lookalike',
        'campaign_period_food_total_lookalike', 'post_period_orders_lookalike',
    ]].mean().reset_index()
    grouped_df = grouped_df.rename(columns={'base_segment_treatment': 'base_segment'})

    # KNN for unmatched
    missing_ids = set(treatment['customerid']) - set(grouped_df['customerid_treatment'])
    unmatched_df = treatment[treatment['customerid'].isin(missing_ids)].copy()

    columns_to_scale = ['optin', 'L14D_orders', 'L30D_orders', 'L90D_orders',
                        'L180D_orders', 'L365D_orders', 'L365D_AOV_cat']

    if not unmatched_df.empty and not lookalike.empty:
        scaler = MinMaxScaler()
        lookalike_scaled = lookalike.copy()
        unmatched_scaled = unmatched_df.copy()
        lookalike_scaled[columns_to_scale] = scaler.fit_transform(lookalike[columns_to_scale])
        unmatched_scaled[columns_to_scale] = scaler.transform(unmatched_df[columns_to_scale])

        knn = NearestNeighbors(n_neighbors=1)
        knn.fit(lookalike_scaled[columns_to_scale])
        distances, indices = knn.kneighbors(unmatched_scaled[columns_to_scale])

        nearest_neighbors_df = pd.DataFrame({
            'customerid_treatment': unmatched_scaled['customerid'].values,
            'country': country,
            'optin': unmatched_scaled['optin'].values,
            'campaign_start_date': pd.to_datetime(date_str).date(),
            'L30D_orders_treatment': unmatched_df['L30D_orders'].values,
            'L30D_GMV_treatment': unmatched_df['L30D_GMV'].values,
            'campaign_period_orders_treatment': unmatched_scaled['campaign_period_orders'].values,
            'post_period_orders_treatment': unmatched_scaled['post_period_orders'].values,
            'base_segment': unmatched_scaled['base_segment'].values,
            'L30D_orders_lookalike': lookalike.loc[lookalike_scaled.iloc[indices.flatten()].index, 'L30D_orders'].values,
            'L30D_GMV_lookalike': lookalike.loc[lookalike_scaled.iloc[indices.flatten()].index, 'L30D_GMV'].values,
            'campaign_period_orders_lookalike': lookalike_scaled['campaign_period_orders'].iloc[indices.flatten()].values,
            'post_period_orders_lookalike': lookalike_scaled['post_period_orders'].iloc[indices.flatten()].values,
        })
    else:
        nearest_neighbors_df = pd.DataFrame()

    grouped_df['source'] = 'exact_match'
    if not nearest_neighbors_df.empty:
        nearest_neighbors_df['source'] = 'nearest_neighbor'
        final_df = pd.concat([grouped_df.reset_index(drop=True),
                              nearest_neighbors_df.reset_index(drop=True)],
                             ignore_index=True)
    else:
        final_df = grouped_df.reset_index(drop=True)

    final_df['campaign_name'] = campaign_name
    final_df['variant'] = 'V2'
    final_df['run_timestamp'] = pd.Timestamp.now(tz='Europe/Amsterdam')
    return final_df

---
## 7. Orchestrator — Run All Windows and Seeds

In [ ]:
def run_all_aa_tests(client, countries, time_windows, n_seeds,
                     treatment_share, audience_table, look_alike_conditions,
                     segment_col, segments_filter):
    """
    Main orchestrator: for each country x window x seed,
    create audience, upload, run V3 matching, collect results.
    """
    all_results = []
    total_runs = len(countries) * len(time_windows) * n_seeds
    run_count = 0

    for country in countries:
        for window in time_windows:
            for seed in range(n_seeds):
                run_count += 1
                campaign_name = f"{AA_CAMPAIGN_PREFIX}_{country}_{window['start']}_seed{seed}"
                print(f'\n=== Run {run_count}/{total_runs}: {campaign_name} ===')

                try:
                    # Step 1: Create audience
                    df_audience, cname = create_aa_audience(
                        client, country, window, seed,
                        treatment_share, segment_col, segments_filter
                    )

                    # Step 2: Upload to BQ
                    upload_aa_audience(client, df_audience, audience_table)

                    # Step 3: Run V3 matching
                    result_df = run_matching_v3(
                        client, country, window['start'], cname,
                        audience_table, look_alike_conditions
                    )

                    if result_df is not None and len(result_df) > 0:
                        result_df['seed'] = seed
                        result_df['window_start'] = window['start']
                        result_df['window_end'] = window['end']
                        result_df['variant'] = 'V3'
                        all_results.append(result_df)
                    else:
                        print(f'  WARNING: No results for {cname}')

                except Exception as e:
                    print(f'  ERROR in {campaign_name}: {e}')

                finally:
                    # Step 4: Cleanup audience rows
                    try:
                        cleanup_aa_audience(client, audience_table, campaign_name)
                    except Exception:
                        pass  # Table may not exist yet

    if all_results:
        return pd.concat(all_results, ignore_index=True)
    else:
        print('ERROR: No results collected from any run.')
        return pd.DataFrame()

In [ ]:
print('Starting A/A test runs...')
start_time = time.time()

df_all_results = run_all_aa_tests(
    client=client,
    countries=COUNTRIES,
    time_windows=TIME_WINDOWS,
    n_seeds=N_SEEDS,
    treatment_share=TREATMENT_SHARE,
    audience_table=AA_AUDIENCE_TABLE,
    look_alike_conditions=LOOK_ALIKE_CONDITIONS_V3,
    segment_col=SEGMENT_COLUMN,
    segments_filter=SEGMENTS_TO_TEST,
)

elapsed = (time.time() - start_time) / 60
print(f'\n{"=" * 50}')
print(f'All runs complete in {elapsed:.1f} minutes.')
print(f'Total result rows: {len(df_all_results)}')

---
## 8. Bias Metrics Calculation

In [ ]:
def calculate_bias_metrics(df, group_cols):
    """
    Calculate uplift and incremental order metrics.

    Uses the same formulas as the production results query:
    uplift = SUM(treatment) / SUM(lookalike) - 1
    """
    agg = df.groupby(group_cols).agg(
        n_customers=('customerid_treatment', 'nunique'),
        n_exact=('source', lambda x: (x == 'exact_match').sum()),
        n_knn=('source', lambda x: (x == 'nearest_neighbor').sum()),
        L30D_treatment=('L30D_orders_treatment', 'sum'),
        L30D_lookalike=('L30D_orders_lookalike', 'sum'),
        campaign_treatment=('campaign_period_orders_treatment', 'sum'),
        campaign_lookalike=('campaign_period_orders_lookalike', 'sum'),
        campaign_promo_treatment=('campaign_period_promo_orders_treatment', 'sum'),
        campaign_promo_lookalike=('campaign_period_promo_orders_lookalike', 'sum'),
        post_treatment=('post_period_orders_treatment', 'sum'),
        post_lookalike=('post_period_orders_lookalike', 'sum'),
    ).reset_index()

    # Uplift = treatment / lookalike - 1 (using safe division)
    agg['pre_period_uplift'] = np.where(
        agg['L30D_lookalike'] > 0,
        agg['L30D_treatment'] / agg['L30D_lookalike'] - 1,
        np.nan
    )
    agg['campaign_period_uplift'] = np.where(
        agg['campaign_lookalike'] > 0,
        agg['campaign_treatment'] / agg['campaign_lookalike'] - 1,
        np.nan
    )
    agg['post_period_uplift'] = np.where(
        agg['post_lookalike'] > 0,
        agg['post_treatment'] / agg['post_lookalike'] - 1,
        np.nan
    )

    # Incremental orders
    agg['campaign_incr_orders'] = agg['campaign_treatment'] - agg['campaign_lookalike']
    agg['post_incr_orders'] = agg['post_treatment'] - agg['post_lookalike']

    # Exact match coverage
    agg['exact_match_pct'] = np.where(
        agg['n_customers'] > 0,
        agg['n_exact'] / agg['n_customers'],
        np.nan
    )

    # V3 promo correction factor (production lines 385-387)
    promo_share_lookalike = np.where(
        agg['campaign_lookalike'] > 0,
        agg['campaign_promo_lookalike'] / agg['campaign_lookalike'],
        0
    )
    promo_share_treatment = np.where(
        agg['campaign_treatment'] > 0,
        agg['campaign_promo_treatment'] / agg['campaign_treatment'],
        0
    )
    agg['incr_correction_factor'] = np.where(
        promo_share_treatment > 0,
        promo_share_lookalike / promo_share_treatment,
        np.nan
    )

    return agg


# Calculate metrics at TOTAL level (per run)
total_group = ['window_start', 'seed', 'country', 'variant']
df_metrics_total = calculate_bias_metrics(df_all_results, total_group)
df_metrics_total['base_segment'] = 'TOTAL'

print(f'Total-level metrics: {len(df_metrics_total)} runs')
print(f'Mean campaign uplift: {df_metrics_total["campaign_period_uplift"].mean():.4f}')
print(f'Mean post uplift: {df_metrics_total["post_period_uplift"].mean():.4f}')

# Calculate metrics at SEGMENT level (per run x segment)
segment_group = ['window_start', 'seed', 'country', 'variant', 'base_segment']
df_metrics_segment = calculate_bias_metrics(df_all_results, segment_group)

print(f'\nSegment-level metrics: {len(df_metrics_segment)} rows')
print('Mean campaign uplift by segment:')
print(df_metrics_segment.groupby('base_segment')['campaign_period_uplift'].mean().sort_values())

# Combine for downstream analysis
df_metrics = pd.concat([df_metrics_total, df_metrics_segment], ignore_index=True)

---
## 9. Statistical Validation

In [ ]:
def statistical_validation(metrics_df, segment='TOTAL'):
    """
    Test whether observed uplift is statistically different from zero.

    1. One-sample t-test: H0: mean(uplift) = 0
    2. Bootstrap 95% CI on mean uplift
    """
    df_seg = metrics_df[metrics_df['base_segment'] == segment]
    results = {}

    for period in ['pre_period_uplift', 'campaign_period_uplift', 'post_period_uplift']:
        values = df_seg[period].dropna().values
        if len(values) < 3:
            results[period] = {
                'mean': np.nan, 'std': np.nan, 'n': len(values),
                't_stat': np.nan, 'p_value': np.nan,
                'ci_lower': np.nan, 'ci_upper': np.nan,
                'ci_contains_zero': None,
            }
            continue

        # One-sample t-test
        t_stat, p_value = stats.ttest_1samp(values, 0)

        # Bootstrap 95% CI
        rng = np.random.RandomState(42)
        boot_means = [
            rng.choice(values, size=len(values), replace=True).mean()
            for _ in range(10000)
        ]
        ci_lower = np.percentile(boot_means, 2.5)
        ci_upper = np.percentile(boot_means, 97.5)

        results[period] = {
            'mean': values.mean(),
            'std': values.std(),
            'n': len(values),
            't_stat': t_stat,
            'p_value': p_value,
            'ci_lower': ci_lower,
            'ci_upper': ci_upper,
            'ci_contains_zero': (ci_lower <= 0 <= ci_upper),
        }

    return pd.DataFrame(results).T


# Total-level validation
print('=== TOTAL-LEVEL STATISTICAL VALIDATION ===')
df_stats_total = statistical_validation(df_metrics, 'TOTAL')
display(df_stats_total)

# Per-segment validation
print('\n=== PER-SEGMENT STATISTICAL VALIDATION ===')
segment_stats = {}
for seg in df_metrics['base_segment'].unique():
    if seg == 'TOTAL':
        continue
    seg_stats = statistical_validation(df_metrics, seg)
    seg_stats['base_segment'] = seg
    segment_stats[seg] = seg_stats

df_stats_segments = pd.concat(segment_stats.values(), ignore_index=False)
display(df_stats_segments[['base_segment', 'mean', 'p_value', 'ci_lower', 'ci_upper', 'ci_contains_zero']])

---
## 10. Pass/Fail Assessment

In [ ]:
def assess_pass_fail(stats_df, segment_name, threshold_hard, threshold_warn):
    """Apply pass/fail criteria to statistical results."""
    verdicts = {}
    for period in ['pre_period_uplift', 'campaign_period_uplift', 'post_period_uplift']:
        if period not in stats_df.index:
            continue
        row = stats_df.loc[period]
        abs_mean = abs(row['mean']) if not np.isnan(row['mean']) else np.nan

        if np.isnan(abs_mean):
            verdict = 'INSUFFICIENT_DATA'
        elif abs_mean > threshold_hard:
            verdict = 'HARD_FAIL'
        elif abs_mean > threshold_warn:
            verdict = 'WARNING'
        elif row['ci_contains_zero'] is False:
            verdict = 'WARNING'
        else:
            verdict = 'PASS'

        verdicts[period] = {
            'segment': segment_name,
            'abs_mean_uplift': abs_mean,
            'p_value': row['p_value'],
            'ci_contains_zero': row['ci_contains_zero'],
            'verdict': verdict,
        }

    return pd.DataFrame(verdicts).T


# Assess TOTAL level
print('=' * 60)
print('TOTAL-LEVEL VERDICT')
print('=' * 60)
df_verdict_total = assess_pass_fail(
    df_stats_total, 'TOTAL', BIAS_THRESHOLD_HARD, BIAS_THRESHOLD_WARN
)
display(df_verdict_total)

overall_verdict = 'PASS'
if 'HARD_FAIL' in df_verdict_total['verdict'].values:
    overall_verdict = 'HARD_FAIL'
elif 'WARNING' in df_verdict_total['verdict'].values:
    overall_verdict = 'WARNING'

print(f'\nOVERALL A/A TEST VERDICT: {overall_verdict}')

# Assess per segment
print('\n' + '=' * 60)
print('PER-SEGMENT VERDICTS')
print('=' * 60)
all_segment_verdicts = []
for seg in sorted(df_metrics['base_segment'].unique()):
    if seg == 'TOTAL':
        continue
    seg_stats = statistical_validation(df_metrics, seg)
    seg_verdict = assess_pass_fail(
        seg_stats, seg, BIAS_THRESHOLD_HARD, BIAS_THRESHOLD_WARN
    )
    all_segment_verdicts.append(seg_verdict)

if all_segment_verdicts:
    df_all_verdicts = pd.concat(all_segment_verdicts)
    display(df_all_verdicts[['segment', 'abs_mean_uplift', 'verdict']].pivot_table(
        index='segment', columns=df_all_verdicts.index, values='verdict', aggfunc='first'
    ))

---
## 11. Visualisations

In [ ]:
# 11.1 Histogram: uplift distribution across all iterations
df_total = df_metrics[df_metrics['base_segment'] == 'TOTAL']

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
periods = ['pre_period_uplift', 'campaign_period_uplift', 'post_period_uplift']
titles = ['Pre-Period Uplift', 'Campaign-Period Uplift', 'Post-Period Uplift']

for i, (period, title) in enumerate(zip(periods, titles)):
    ax = axes[i]
    values = df_total[period].dropna()
    sns.histplot(values, kde=True, ax=ax, bins=20, color='steelblue')
    ax.axvline(x=0, color='red', linestyle='--', linewidth=2, label='Expected (0)')
    ax.axvline(x=values.mean(), color='orange', linestyle='-',
               linewidth=2, label=f'Mean: {values.mean():.4f}')
    ax.set_title(title, fontsize=13)
    ax.set_xlabel('Uplift')
    ax.legend(fontsize=9)

plt.suptitle('A/A Test: Uplift Distribution (Total Level)', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# 11.2 Box plot: uplift by time window (stability)
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, (period, title) in enumerate(zip(periods, titles)):
    ax = axes[i]
    df_total.boxplot(column=period, by='window_start', ax=ax)
    ax.axhline(y=0, color='red', linestyle='--', linewidth=1.5)
    ax.set_title(title, fontsize=13)
    ax.set_xlabel('Time Window')
    ax.set_ylabel('Uplift')
    plt.sca(ax)
    plt.xticks(rotation=45)

plt.suptitle('A/A Test: Uplift Stability Across Time Windows', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# 11.3 Incremental orders distribution (should center at 0)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for i, (col, title) in enumerate([
    ('campaign_incr_orders', 'Campaign Incremental Orders'),
    ('post_incr_orders', 'Post-Period Incremental Orders'),
]):
    ax = axes[i]
    values = df_total[col].dropna()
    sns.histplot(values, kde=True, ax=ax, bins=20, color='coral')
    ax.axvline(x=0, color='red', linestyle='--', linewidth=2)
    ax.axvline(x=values.mean(), color='orange', linestyle='-',
               linewidth=2, label=f'Mean: {values.mean():.0f}')
    ax.set_title(title, fontsize=13)
    ax.set_xlabel('Incremental Orders')
    ax.legend()

plt.suptitle('A/A Test: Incremental Orders (should be ~0)', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# 11.4 Exact match coverage distribution
fig, ax = plt.subplots(figsize=(8, 5))
sns.histplot(df_total['exact_match_pct'].dropna(), kde=True, ax=ax, bins=20, color='seagreen')
ax.set_title('Exact Match Coverage Across Runs', fontsize=13)
ax.set_xlabel('Exact Match %')
ax.axvline(x=df_total['exact_match_pct'].mean(), color='orange', linestyle='-',
           linewidth=2, label=f'Mean: {df_total["exact_match_pct"].mean():.2%}')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# 11.5 Heatmap: bias by segment
df_seg = df_metrics[df_metrics['base_segment'] != 'TOTAL']
if len(df_seg) > 0:
    heatmap_data = df_seg.groupby('base_segment')[periods].mean()

    fig, ax = plt.subplots(figsize=(10, 7))
    sns.heatmap(
        heatmap_data, annot=True, fmt='.4f', cmap='RdYlGn_r',
        center=0, ax=ax, linewidths=0.5,
        xticklabels=['Pre-Period', 'Campaign', 'Post-Period']
    )
    ax.set_title('Mean Uplift by Segment (0 = no bias)', fontsize=13)
    ax.set_ylabel('Base Segment')
    plt.tight_layout()
    plt.show()
else:
    print('No segment-level data to plot.')

In [ ]:
# 11.6 Box plot: campaign uplift by segment
if len(df_seg) > 0:
    fig, ax = plt.subplots(figsize=(12, 6))
    order = sorted(df_seg['base_segment'].unique())
    sns.boxplot(
        data=df_seg, x='base_segment', y='campaign_period_uplift',
        order=order, ax=ax, palette='Set2'
    )
    ax.axhline(y=0, color='red', linestyle='--', linewidth=1.5)
    ax.axhline(y=BIAS_THRESHOLD_WARN, color='orange', linestyle=':', alpha=0.7,
               label=f'Warning threshold ({BIAS_THRESHOLD_WARN:.0%})')
    ax.axhline(y=-BIAS_THRESHOLD_WARN, color='orange', linestyle=':', alpha=0.7)
    ax.set_title('Campaign-Period Uplift by Segment', fontsize=13)
    ax.set_xlabel('Base Segment')
    ax.set_ylabel('Uplift')
    ax.legend()
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

---
## 12. Stability Analysis

In [ ]:
def stability_analysis(metrics_df):
    """Assess whether bias is consistent across windows or varies."""
    df_t = metrics_df[metrics_df['base_segment'] == 'TOTAL']

    # Cross-window mean and std
    window_stats = df_t.groupby('window_start').agg({
        'campaign_period_uplift': ['mean', 'std', 'count'],
        'post_period_uplift': ['mean', 'std'],
        'exact_match_pct': ['mean'],
    })
    print('Cross-window statistics:')
    display(window_stats)

    # Levene test for equal variance across windows
    groups = [
        g['campaign_period_uplift'].dropna().values
        for _, g in df_t.groupby('window_start')
    ]
    groups = [g for g in groups if len(g) >= 2]

    if len(groups) > 1:
        levene_stat, levene_p = stats.levene(*groups)
        print(f'\nLevene test for equal variance across windows:')
        print(f'  Statistic: {levene_stat:.4f}, p-value: {levene_p:.4f}')
        if levene_p < 0.05:
            print('  WARNING: Variance differs significantly across windows.')
            print('  The methodology may be unstable — bias depends on the time period.')
        else:
            print('  PASS: Variance is consistent across windows.')

        # Kruskal-Wallis test for equal medians across windows
        kw_stat, kw_p = stats.kruskal(*groups)
        print(f'\nKruskal-Wallis test for equal medians across windows:')
        print(f'  Statistic: {kw_stat:.4f}, p-value: {kw_p:.4f}')
        if kw_p < 0.05:
            print('  WARNING: Median uplift differs across windows.')
        else:
            print('  PASS: Median uplift is consistent across windows.')

    return window_stats


window_stats = stability_analysis(df_metrics)

---
## 13. Summary Report

In [ ]:
print('=' * 60)
print('A/A TEST SUMMARY REPORT')
print('=' * 60)

print(f'\nConfiguration:')
print(f'  Markets:           {COUNTRIES}')
print(f'  Time windows:      {len(TIME_WINDOWS)}')
print(f'  Seeds per window:  {N_SEEDS}')
print(f'  Total runs:        {len(df_metrics[df_metrics["base_segment"]=="TOTAL"])}')
print(f'  Segment filter:    {SEGMENTS_TO_TEST or "all"}')
print(f'  Variant:           V3 (with promo correction)')

print(f'\nThresholds:')
print(f'  Hard fail:  |uplift| > {BIAS_THRESHOLD_HARD:.0%}')
print(f'  Warning:    |uplift| > {BIAS_THRESHOLD_WARN:.0%}')

df_total = df_metrics[df_metrics['base_segment'] == 'TOTAL']
print(f'\nTotal-Level Results (across {len(df_total)} runs):')
for period, label in zip(periods, titles):
    vals = df_total[period].dropna()
    if len(vals) > 0:
        print(f'  {label}:')
        print(f'    Mean uplift:   {vals.mean():+.4f} ({vals.mean():+.2%})')
        print(f'    Std:           {vals.std():.4f}')
        print(f'    Range:         [{vals.min():.4f}, {vals.max():.4f}]')

print(f'\nExact match coverage: {df_total["exact_match_pct"].mean():.1%} '
      f'(std: {df_total["exact_match_pct"].std():.1%})')

print(f'\n{"=" * 60}')
print(f'OVERALL VERDICT: {overall_verdict}')
print(f'{"=" * 60}')

if overall_verdict == 'PASS':
    print('The Customer Lookalike V3 methodology shows no statistically significant')
    print('bias in the tested time windows. Proceed to Phase 2 optimisation.')
elif overall_verdict == 'WARNING':
    print('The methodology shows minor bias. Review per-segment results to identify')
    print('which segments contribute. Consider excluding high-bias segments.')
else:
    print('The methodology shows significant inherent bias. Do NOT use for production')
    print('evaluation until matching logic is improved (Phase 2).')

---
## 14. Persist Results to BigQuery (Optional)

In [ ]:
PERSIST_RESULTS = False  # Set to True to save results to BigQuery

if PERSIST_RESULTS:
    schema = [
        bigquery.SchemaField('window_start', 'STRING'),
        bigquery.SchemaField('seed', 'INTEGER'),
        bigquery.SchemaField('country', 'STRING'),
        bigquery.SchemaField('variant', 'STRING'),
        bigquery.SchemaField('base_segment', 'STRING'),
        bigquery.SchemaField('n_customers', 'INTEGER'),
        bigquery.SchemaField('exact_match_pct', 'FLOAT'),
        bigquery.SchemaField('pre_period_uplift', 'FLOAT'),
        bigquery.SchemaField('campaign_period_uplift', 'FLOAT'),
        bigquery.SchemaField('post_period_uplift', 'FLOAT'),
        bigquery.SchemaField('campaign_incr_orders', 'FLOAT'),
        bigquery.SchemaField('post_incr_orders', 'FLOAT'),
    ]

    persist_cols = [
        'window_start', 'seed', 'country', 'variant', 'base_segment',
        'n_customers', 'exact_match_pct', 'pre_period_uplift',
        'campaign_period_uplift', 'post_period_uplift',
        'campaign_incr_orders', 'post_incr_orders',
    ]

    job_config = bigquery.LoadJobConfig(
        schema=schema,
        write_disposition='WRITE_APPEND',
    )

    job = client.load_table_from_dataframe(
        df_metrics[persist_cols], AA_RESULTS_TABLE, job_config=job_config
    )
    job.result()
    print(f'Persisted {len(df_metrics)} rows to {AA_RESULTS_TABLE}')
else:
    print('Results not persisted. Set PERSIST_RESULTS = True to save to BigQuery.')